# BirdCLEF2026 - Evaluation and Submission

This notebook handles model evaluation, inference, and submission file generation.

## Why This Step is Critical

1. **Test data is different** - Field recordings from Pantanal
2. **Need proper preprocessing** - Must match training exactly
3. **Submission format** - 12 segments per file, 234 species
4. **No test files locally** - Will be populated on Kaggle

This notebook handles the domain shift challenge!

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Set paths
DATA_DIR = Path("..")
TEST_SOUNDSCAPES_DIR = DATA_DIR / "test_soundscapes"
OUTPUT_DIR = Path("output")
MODELS_DIR = OUTPUT_DIR / "models"

# Configuration (must match training)
SAMPLE_RATE = 32000
DURATION = 5
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
EXPECTED_SAMPLES = SAMPLE_RATE * DURATION

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Configuration and Models

Load saved configuration and trained model weights.

In [ ]:
# Load submission template
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
species_columns = [col for col in sample_submission.columns if col not in ['row_id', 'filename', 'end_time']]
species_to_idx = {sp: i for i, sp in enumerate(species_columns)}
idx_to_species = {i: sp for sp, i in species_to_idx.items()}
NUM_CLASSES = len(species_columns)

# Load species mapping
with open(OUTPUT_DIR / 'species_mapping.json', 'r') as f:
    saved_species_to_idx = json.load(f)
species_to_idx = {k: int(v) for k, v in saved_species_to_idx.items()}

print(f"Number of classes: {NUM_CLASSES}")
print(f"Species columns: {len(species_columns)}")

In [ ]:
import timm
import librosa
import soundfile as sf

# Define model (same architecture as training)
class BirdAudioClassifier(nn.Module):
    """EfficientNet-based classifier (same as training)."""
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.num_features = self.backbone.num_features
        self.classifier = nn.Linear(self.num_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.classifier(features)
        return out

# Load trained model
model = BirdAudioClassifier('efficientnet_b0', NUM_CLASSES, pretrained=False)
model_path = MODELS_DIR / 'efficientnet_best.pt'
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print(f"Loaded model from {model_path}")
else:
    print("No trained model found. Using pretrained model without fine-tuning.")
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(device)
    model.eval()

## 2. Audio Processing Functions

Must match training preprocessing exactly.

In [ ]:
# Audio functions (must match training exactly)
def load_audio(file_path, target_sr=SAMPLE_RATE):
    """Load audio file and resample."""
    try:
        waveform, sr = sf.read(file_path)
        if len(waveform.shape) > 1:
            waveform = waveform.mean(axis=1)
        if sr != target_sr:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
        return waveform, target_sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

def get_mel_spectrogram(waveform, sr=SAMPLE_RATE):
    """Generate mel spectrogram."""
    mel_spec = librosa.feature.melspectrogram(
        y=waveform, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=50, fmax=15000, power=2.0
    )
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel_spec

def normalize_spectrogram(spec):
    """Normalize to [0, 1]."""
    spec = spec - spec.min()
    spec = spec / (spec.max() + 1e-8)
    return spec

print("Audio processing functions defined.")

## 3. Process Test Soundscapes

### Why Process This Way?

- 1-minute test files → 12 segments of 5 seconds each
- Each segment needs prediction for all 234 species
- Test files will be populated when running on Kaggle

In [ ]:
# Get test files
test_files = sorted(TEST_SOUNDSCAPES_DIR.glob("*.ogg"))
print(f"Found {len(test_files)} test soundscape files")

if len(test_files) == 0:
    print("\n⚠️ No test files found in test_soundscapes directory.")
    print("Note: Test data will be populated when notebook is run on Kaggle.")
    print("The submission code below will handle this case.")
    test_files = []

In [ ]:
def process_test_file(file_path, device, model):
    """Process a single test soundscape file and return predictions for all segments.
    
    Processing:
    1. Load 1-minute audio file
    2. Split into 12 non-overlapping 5-second segments
    3. Generate spectrogram for each segment
    4. Get prediction probabilities for all 234 species
    
    Returns:
        List of 12 prediction arrays, each of shape (234,)
    """
    # Load audio
    waveform, sr = load_audio(file_path)
    if waveform is None:
        return None
    
    # Handle stereo
    if len(waveform.shape) > 1:
        waveform = waveform.mean(axis=1)
    
    # Resample if needed
    if sr != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)
    
    # Get predictions for each 5-second segment
    # 1-minute file = 12 segments (5, 10, 15, ..., 60)
    predictions = []
    
    for end_time in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]:
        start_sample = (end_time - 5) * SAMPLE_RATE
        end_sample = end_time * SAMPLE_RATE
        
        segment = waveform[start_sample:end_sample]
        
        # Pad if needed
        if len(segment) < EXPECTED_SAMPLES:
            segment = np.pad(segment, (0, EXPECTED_SAMPLES - len(segment)), mode='constant')
        
        # Get spectrogram
        spec = get_mel_spectrogram(segment, SAMPLE_RATE)
        spec = normalize_spectrogram(spec)
        spec = torch.FloatTensor(spec).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
        
        # Predict
        with torch.no_grad():
            spec = spec.to(device)
            output = model(spec)
            pred = torch.sigmoid(output).cpu().numpy()[0]
        
        predictions.append(pred)
    
    return predictions

print("Test processing function defined.")

## 4. Generate Predictions

Process all test files and collect predictions.

In [ ]:
if len(test_files) > 0:
    all_predictions = {}
    
    for test_file in tqdm(test_files, desc="Processing test files"):
        predictions = process_test_file(test_file, device, model)
        if predictions:
            filename = test_file.name
            all_predictions[filename] = predictions
    
    print(f"Processed {len(all_predictions)} test files")
else:
    print("No test files to process.")
    all_predictions = {}

## 5. Create Submission

### Submission Format

- Row ID: `{filename}_{end_time}`
- Columns: row_id + 234 species
- Values: Probability [0, 1] for each species


In [ ]:
# Parse sample submission to get expected row_ids
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
submission = sample_submission.copy()

print(f"Submission shape: {submission.shape}")
print(f"Sample submission rows: {len(submission)}")

In [ ]:
# Function to create submission from predictions
def create_submission(predictions_dict, sample_submission_df, species_columns):
    """Create submission dataframe from predictions.
    
    For each row in sample_submission:
    1. Parse filename and end_time from row_id
    2. Look up predictions for that file/segment
    3. Fill in species probabilities
    
    If no predictions available, use prior probability.
    """
    submission = sample_submission_df.copy()
    
    # Initialize with small probability (prior)
    prior = 0.01
    for col in species_columns:
        submission[col] = prior
    
    # Fill in predictions
    for row_idx, row in tqdm(submission.iterrows(), total=len(submission), desc="Creating submission"):
        row_id = row['row_id']
        
        # Parse row_id
        parts = row_id.split('_')
        filename = '_'.join(parts[:-1]) + '.ogg'
        end_time = int(parts[-1])
        
        # Get segment index (end_time // 5 - 1)
        segment_idx = end_time // 5 - 1
        
        if filename in predictions_dict:
            preds = predictions_dict[filename][segment_idx]
            
            for i, species in enumerate(species_columns):
                if i < len(preds):
                    submission.loc[row_idx, species] = preds[i]
    
    return submission

# If we have predictions, create submission
if len(all_predictions) > 0:
    submission = create_submission(all_predictions, sample_submission, species_columns)
else:
    # Create baseline submission with uniform probabilities
    print("Creating baseline submission (no test files available locally)...")
    print("This will be overwritten when running on Kaggle with actual test data.")
    submission = sample_submission.copy()
    
    # Fill with prior probability
    prior = 1.0 / NUM_CLASSES  # Uniform probability
    for col in species_columns:
        submission[col] = prior
    
    print(f"Baseline submission created with prior: {prior:.6f}")

In [ ]:
# Check submission
print("\nSubmission preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"\nPrediction statistics:")
for col in species_columns[:5]:
    print(f"  {col}: mean={submission[col].mean():.4f}, min={submission[col].min():.4f}, max={submission[col].max():.4f}")

## 6. Save Submission

In [ ]:
# Save submission
submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")

# Verify submission
verify_submission = pd.read_csv(submission_path)
print(f"\nVerification:")
print(f"  Shape: {verify_submission.shape}")
print(f"  Columns: {len(verify_submission.columns)}")
print(f"  All probabilities in [0, 1]: {(verify_submission[species_columns].values >= 0).all() and (verify_submission[species_columns].values <= 1).all()}")

## Summary

This notebook:

1. **Loaded trained model** - EfficientNet-B0 with best validation AUC
2. **Processed test soundscapes** - 12 segments per 1-minute file
3. **Generated predictions** - Probability for each of 234 species
4. **Created submission** - Matches sample_submission format
5. **Saved submission.csv** - Ready for Kaggle submission

**Note**: When running on Kaggle, the test files will be populated and actual predictions will be generated.